In [ ]:
from openadmet.toolkit.database.chembl import Caco2ChEMBLCurator, MDCKChEMBLCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm

# Curating Caco-2 permeability data from ChEMBL and pushing to a remote intake catalog

Our goal is to curate activity data from ChEMBL and push this to a remote location with a catalog that can be used by others to look up our data. This will enable consistency and rapid dissemination of our work as well as an over-time evolution of our data curation practices. 

We use the `Intake` package for a lightweight self-describing data parsing workflow. Read more about intake here: https://intake.readthedocs.io/en/latest/index.html


Here we gather `a_to_b` and `b_to_a` data permissivley from ChEMBL (ie without activity based curation). 
Caco2 has BAO BAO_0000219 (inhibition of TEA uptake in OCT2-expressing HEK293 cells) and target CHEMBL1743122.

We then aggregate  measurements on the same compound by taking the mean and median. This is the most basic form of curation available, but serves as a good baseline for our initial models. 



A to B (in) is different to B to A (out) because the membrane is not "symmetric". Ratio can tell us whether the transport is active

## gather ChEMBL data

First we need to gather in our data from ChEMBL using our SQL API defined in `openadmet-toolkit`

We use `OPENADMET_CANONICAL_SMILES` and `OPENADMET_INCHIKEY` to distinguish our ML ready representation from the source SMILES

Below, we define functions for curating different data types: A->B, B->A and all (no curation)

In [2]:
def gather_chembl_data_a_to_b(chembl_ver: int):
    print(f"working on target")
    pctc = Caco2ChEMBLCurator( version=chembl_ver, a_to_b=True)
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data
        

In [3]:
def gather_chembl_data_b_to_a(chembl_ver: int):
    print(f"working on target")
    pctc = Caco2ChEMBLCurator( version=chembl_ver, b_to_a=True)
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data

In [4]:
def gather_chembl_data_all_caco(chembl_ver: int):
    print(f"working on target")
    pctc = Caco2ChEMBLCurator( version=chembl_ver )
    activity_data = pctc.get_activity_data(return_as="df")
    print("canonicalising raw data")
    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data

In [5]:
chembl_ver = 35

In [6]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

# Setup S3

After curating our data we would like to push to a remote bucket to save both the raw data and the catalog

In [8]:
settings = S3Settings()

In [9]:
bucket = "openadmet-data-public-dev"

In [10]:
bucket = S3Bucket.from_settings(settings, bucket)

In [11]:
import datetime

In [12]:
t = datetime.datetime.now()

In [13]:
date = t.strftime("%Y-%m-%d")

In [14]:
# keep the same location for all Caco-2 data
location=f"ChEMBL{chembl_ver}_Caco2_permeability"

In [15]:
import os
from pathlib import Path

location_path = Path(location)

In [16]:
location_path.mkdir(exist_ok=True)

We can just use a one off here as no need to loop over targets e.g in activity curation

In [17]:
uris_raw = {}
uris_agg = {}
for data_types in ["a_to_b", "b_to_a", "all_caco"]:
    target = f"caco2_{data_types}"
    if data_types == "a_to_b":
        agg, raw = gather_chembl_data_a_to_b(chembl_ver)
    elif data_types == "b_to_a":
        agg, raw = gather_chembl_data_b_to_a(chembl_ver)
    else:
        agg, raw = gather_chembl_data_all_caco(chembl_ver)

    fname_agg = f"ChEMBL_Caco2_{target}_aggregated.parquet"
    fname_raw = f"ChEMBL_Caco2_{target}_raw.parquet"
        
    agg.to_parquet(location_path/fname_agg)
    raw.to_parquet(location_path/fname_raw)
        
    bucket_destination_agg = location + "/" + fname_agg
    bucket.push_file(location_path/fname_agg, bucket_destination_agg)
    bucket_destination_raw = location + "/" + fname_raw
    bucket.push_file(location_path/fname_raw, bucket_destination_raw)

    # get S3 URIs
    uri_agg = bucket.to_uri(bucket_destination_agg)
    uris_agg[target] = uri_agg

    uri_raw = bucket.to_uri(bucket_destination_raw)
    uris_raw[target] = uri_raw

working on target
canonicalising raw data


100%|██████████| 3818/3818 [00:00<00:00, 5059.13it/s]


smiles duplicates 0
inchikey duplicates 0
working on target
canonicalising raw data


100%|██████████| 1766/1766 [00:00<00:00, 4859.27it/s]


smiles duplicates 0
inchikey duplicates 0
working on target
canonicalising raw data


100%|██████████| 7040/7040 [00:01<00:00, 5021.01it/s]


smiles duplicates 0
inchikey duplicates 0


# Build the Intake Catalog

We have sucessfully aggregted our data and pushed it to a remote destination. Now for others to consume our data, we are going to make an `Intake` catalog such that our data can be readily made available. 

The workflow here is drawn from the `creator` walkthrough from the main intake tutorials https://intake.readthedocs.io/en/latest/walkthrough2.html

TODO: add descriptions to the catalog

In [18]:
import intake

In [19]:
intake.Catalog?

Init signature:
intake.Catalog(
    entries: 'Iterable[ReaderDescription] | Mapping | None' = None,
    aliases: 'dict[str, int] | None' = None,
    data: 'Iterable[DataDescription] | Mapping' = None,
    user_parameters: 'dict[str, BaseUserParameter] | None' = None,
    parameter_overrides: 'dict[str, Any] | None' = None,
    metadata: 'dict | None' = None,
)
Docstring:      A collection of data and reader descriptions.
File:           ~/miniforge3/envs/admet-env/lib/python3.11/site-packages/intake/readers/entry.py
Type:           type
Subclasses:     THREDDSCatalog

In [20]:
cat = intake.entry.Catalog()

In [21]:
uris_agg

{'caco2_a_to_b': 's3://openadmet-data-public-dev/ChEMBL35_Caco2_permeability/ChEMBL_Caco2_caco2_a_to_b_aggregated.parquet',
 'caco2_b_to_a': 's3://openadmet-data-public-dev/ChEMBL35_Caco2_permeability/ChEMBL_Caco2_caco2_b_to_a_aggregated.parquet',
 'caco2_all_caco': 's3://openadmet-data-public-dev/ChEMBL35_Caco2_permeability/ChEMBL_Caco2_caco2_all_caco_aggregated.parquet'}

In [22]:
uris_raw

{'caco2_a_to_b': 's3://openadmet-data-public-dev/ChEMBL35_Caco2_permeability/ChEMBL_Caco2_caco2_a_to_b_raw.parquet',
 'caco2_b_to_a': 's3://openadmet-data-public-dev/ChEMBL35_Caco2_permeability/ChEMBL_Caco2_caco2_b_to_a_raw.parquet',
 'caco2_all_caco': 's3://openadmet-data-public-dev/ChEMBL35_Caco2_permeability/ChEMBL_Caco2_caco2_all_caco_raw.parquet'}

In [23]:
for k,v in uris_agg.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

In [24]:
for k,v in uris_raw.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

## Push the Catalog

Ok now we have made the catalog, lets push it to the remote location so it can live alongside the data. 

The catalog can then be used from S3 or from github etc, anything that exposes a file-like API. 

In [25]:
cat

Catalog
 named datasets: []

In [26]:
catname = f"CATALOG_{location}.yaml"

In [27]:
cat.to_yaml_file(catname)

In [28]:
cat_location = location+ "/" +catname

In [29]:
cat_location

'ChEMBL35_Caco2_permeability/CATALOG_ChEMBL35_Caco2_permeability.yaml'

In [30]:
bucket.push_file(catname, cat_location)

In [31]:
cat_uri = bucket.to_uri(cat_location)

In [32]:
# Now can read the catalog from URI
cat = intake.Catalog.from_yaml_file("s3://openadmet-data-public-dev/ChEMBL35_Caco2_permeability/CATALOG_ChEMBL35_Caco2_permeability.yaml")

In [33]:
cat.entries['caco2_a_to_b_aggregated']

Entry for reader: intake.readers.readers:PandasParquet
  kwargs: {'args': ['{data(4abd22431dedb5a6)}']}
  producing: pandas:DataFrame